# Source gallery: fourteen light-source families, one pipeline

Every source in `highuvlith` implements the same `LithographySource` interface,
so a 436 nm mercury lamp, a 13.5 nm tin plasma and a 6.7 nm gadolinium plasma all
run through the **same** imaging code. This notebook:

1. lists every preset with its wavelength, power and maturity;
2. shows spectra and pupil fills;
3. images the same pattern, at the same $k_1$, through every projection-capable
   preset, which exposes the role of the pupil fill (coherence);
4. closes with the honest per-family status.

> **Honesty first.** These families sit at very different maturity levels: some
> are production tools, some are laboratory sources, some are design projections
> and a few are purely theoretical. Read the `maturity` (the machine) and `status`
> (the model) columns before quoting any number. Notebook 05 goes deeper into the
> source physics and the gaps to high-volume manufacturing (HVM).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import highuvlith as huv
from highuvlith import viz

%matplotlib inline

## 1. Every preset

`huv.source_landscape()` builds each preset from its factory and returns one
record per preset. Numbers are computed live by the factories; `maturity` and
`status` are curated labels taken from the
[capability matrix](../../docs/capability-matrix.md).

Average power is **not** defined at the same plane for every family: plasma
sources (LPP, DPP) report in-band power at the intermediate focus (IF, the
scanner convention), lasers and FELs their output, X-ray tubes the filtered X-ray
power. The `power_definition` column says which.

In [ ]:
presets = huv.source_landscape()
families = sorted({p.family for p in presets} | {"entangled"})
print(f"{len(presets)} presets in {len(families)} families: {', '.join(families)}\n")

def fmt_power(w):
    return "      n/a" if w is None else f"{w:9.3g}"

print(f"{'preset':22s} {'family':14s} {'lambda/nm':>10s} {'P/W':>9s}  {'maturity':12s} {'model':6s}  power definition")
print("-" * 120)
for p in presets:
    print(f"{p.label:22s} {p.family:14s} {p.wavelength_nm:10.4g} {fmt_power(p.average_power_w)}  "
          f"{p.maturity:12s} {p.status:6s}  {p.power_definition[:48]}")

## 2. Spectra and pupil fills

`SourceConfig.spectrum()` lists the spectral samples the engine integrates over
and `pupil_fill()` the illumination in pupil coordinates σ (units of NA/λ).
Excimer and plasma sources are spatially incoherent and fill a large part of the
pupil (σ 0.7 to 0.9). Laser-like sources (HHG, FELs, undulators) are nearly
spatially coherent; they are modeled with a small Gaussian fill whose σ follows
from the source's coherence.

`viz.plot_source` draws both panels for one source.

In [ ]:
for factory in ("f2_laser", "lpp_sn_13nm5", "hhg_ne_13nm5"):
    src = getattr(huv.SourceConfig, factory)()
    fig = viz.plot_source(src)
    fig.suptitle(f"{factory}()  -  {src.wavelength_nm:.4g} nm, fill {src.illumination[0]}", y=1.02)

The hard-X-ray sources (X-ray tube, laser-wakefield betatron) are broadband keV
sources for LIGA and proximity printing, not for projection imaging. Their
relative spectrum comes from `xray_spectrum()`: here a copper tube's Kα and Kβ
lines on the Kramers bremsstrahlung continuum (🔶 continuum shape; the line
energies and the Duane-Hunt cutoff are ✅).

In [ ]:
tube = huv.SourceConfig.xray_tube(anode="Cu")
e_kev, frac = np.array(tube.xray_spectrum(n_bins=400)).T
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(e_kev, frac)
ax.set(xlabel="photon energy (keV)", ylabel="relative photons per bin", yscale="log",
       title=f"Cu X-ray tube, mean wavelength {tube.wavelength_nm:.4f} nm")
ax.grid(alpha=0.3)

## 3. The same pattern through every projection source

To compare sources fairly we fix the normalized half-pitch
$k_1 = (p/2) \mathrm{NA} / \lambda$ and scale the mask with each wavelength. Sources
above 100 nm use a dry refractive projector at NA 0.75; the EUV-range sources use
an unobscured NA 0.33 reflective (NXE-like) pupil. The pupil model is the same
idealized scalar pupil for every wavelength in that group; no multilayer is
assumed to exist at, say, 25 or 46.9 nm.

Two-beam imaging of a grating needs the first diffraction order inside the pupil
for at least part of the source. With a fill of radius σ that happens down to the
pitch $\lambda / (\mathrm{NA} (1 + \sigma))$, i.e. down to $k_1 = 1 / (2 (1 + \sigma))$:

- incoherent fills (σ 0.7 to 0.9) resolve down to $k_1 \approx 0.26$ to $0.29$;
- a nearly coherent fill (σ about 0.05) stops at $k_1 \approx 0.48$.

So at $k_1 = 0.45$ the laser-like sources lose the pattern (the small residual
contrast comes from the tails of their Gaussian fill) while the incoherent
sources still print it. The source is sampled on a fixed 41 × 41 grid for this
comparison.

In [ ]:
def contrast_at_k1(src, k1, spp=41):
    optics = (huv.OpticsConfig(numerical_aperture=0.75) if src.wavelength_nm > 100
              else huv.OpticsConfig.euv_nxe())
    pitch = 2.0 * k1 * src.wavelength_nm / optics.numerical_aperture
    mask = huv.MaskConfig.line_space(pitch / 2.0, pitch)
    grid = mask.commensurate_grid(size=64, target_pixel_nm=pitch / 64)
    engine = huv.SimulationEngine(src, optics, mask, grid=grid, max_kernels=20,
                                  source_points_per_axis=spp)
    return engine.compute_aerial_image().image_contrast()

rows = []
for p in presets:
    if p.family in ("xray_tube", "betatron"):
        continue                                   # LIGA / proximity sources
    src = eval("huv.SourceConfig." + p.factory)
    rows.append((p.label, src.illumination[0], src.sigma_outer,
                 contrast_at_k1(src, 0.45), contrast_at_k1(src, 0.60)))

labels = [r[0] for r in rows]
y = np.arange(len(rows))
fig, ax = plt.subplots(figsize=(8, 9))
ax.barh(y - 0.2, [r[3] for r in rows], height=0.4, label="k1 = 0.45")
ax.barh(y + 0.2, [r[4] for r in rows], height=0.4, label="k1 = 0.60")
ax.set_yticks(y, [f"{r[0]}  (σ {r[2]:.2f})" for r in rows])
ax.invert_yaxis()
ax.set(xlabel="aerial-image contrast", xlim=(0, 1.05),
       title="Equal-k1 line/space through every projection preset")
ax.legend(loc="lower right")
ax.grid(axis="x", alpha=0.3)
fig.tight_layout()

### Off-axis illumination rescues the coherent sources

A coherent source does not have to be used on axis. Reshaping its fill into a
dipole (in a real tool: an illuminator with a pupil facet mirror or a scanned
beam) tilts the illumination so that the zero and first orders straddle the
pupil. `with_illumination` replaces the fill and keeps the spectrum and power.
The poles are placed at $\sigma_c = \lambda / (2 p \mathrm{NA})$, the symmetric
two-beam condition for this pitch.

In [ ]:
hhg = huv.SourceConfig.hhg_ne_13nm5()
k1 = 0.45
pitch = 2 * k1 * hhg.wavelength_nm / 0.33
sigma_c = hhg.wavelength_nm / (2 * pitch * 0.33)
hhg_dipole = hhg.with_illumination("dipole", sigma_c, 0.05, 0.0)
print(f"pitch {pitch:.2f} nm, dipole poles at sigma_c = {sigma_c:.3f}")
print(f"HHG on-axis : contrast {contrast_at_k1(hhg, k1):.3f}")
print(f"HHG dipole  : contrast {contrast_at_k1(hhg_dipole, k1):.3f}")

## 4. Honest per-family status

From the [capability matrix](../../docs/capability-matrix.md), the single source of
truth. ✅ implemented and validated · 🔶 simplified, approximations stated · 🧪
theoretical or research projection. Each family has its own page under
[docs/sources](../../docs/sources/index.md).

| Family | Model status | What is (and is not) modeled |
|---|---|---|
| VUV excimer (F$_2$ 157.63 nm; Ar$_2$ 126 nm) | ✅ (Ar$_2$ 🧪) | Line shape, pupil fill, pulse energy × rep rate, derived photon budget and E95. The Ar$_2$ preset is hypothetical. |
| DUV/UV heritage (ArF, KrF, Hg g/h/i) | ✅ (🔶 parameters) | NIST Hg wavelengths, exact photon budgets; laser bandwidths and powers are representative; lamp power not modeled. |
| LPA-FEL | 🔶 | BELLA 420 nm SASE at 1 Hz is the demonstrated anchor; the 25 nm / 1 kHz preset is a design projection. Bandwidth $2 \rho \lambda$ derived from undulator and beam. |
| Laser-produced plasma (Sn 13.5; Gd 6.7; Tb 6.5 nm) | ✅ (🔶 CE/étendue; Gd/Tb power 🧪) | Drive power × CE × collection × étendue → 250 W at IF on the NXE:3400B chain; 500 W preset; Gd/Tb in-band power is a projection. |
| Discharge-produced plasma (Xe, Sn) | 🔶 | Electrical power → CE → collector → étendue → IF; presets on reported anchors. |
| Synchrotron (undulator, bending magnet) | ✅ (🔶 coherence) | Wavelength derived from machine parameters; sinc$^2$ line; Kim central-cone flux and power. |
| High-harmonic generation | ✅/🔶 | Comb, cutoff law, Keldysh parameter exact; power = assumed driver power × order-of-magnitude efficiency, compared with measured records. |
| XFEL (SASE, seeded) | ✅ (🧪 CW-SC/ERL presets) | Pierce parameter derived from the machine; Gamma statistics drive the dose jitter; the CW-SC and ERL powers are design projections. |
| Soft-X-ray laser (Ar 46.9, Ag 13.9, Cd 13.2, Mo 18.9 nm) | ✅/🔶 | Fixed atomic lasing lines exact; presets reproduce reported average powers; no gain model. |
| X-ray tube (W, Mo, Cu, Rh) | 🔶/✅ | Lines, edges, Duane-Hunt cutoff, Be window ✅; Kramers continuum and absolute flux 🔶. A LIGA source. |
| Inverse Compton scattering | 🧪 | Kinematics exact; derived yield about 71 µW in band, millions of times short of 250 W. |
| Laser-wakefield betatron | 🧪 | Textbook synchrotron-like model; a keV, µW-class imaging source, not a lithography source. |
| SSMB storage ring | 🧪 | Integer-harmonic check live; 1 kW needs a sustained bunching factor near 0.15 (undemonstrated). |
| Smith-Purcell | 🧪 | Dispersion exact; EUV emission is purely theoretical (demonstrated only to about 230 nm). |
| Entangled photons (N00N) | 🧪 | Reports the physical λ; N-photon models are applied explicitly by the quantum module; no N00N sub-Rayleigh pattern has been recorded in a resist. |

Contrast alone says little about a source: at equal $k_1$ every incoherent source
images identically, because the aerial image depends only on λ/NA scaling, the
fill and the spectrum. The real differences are power, coherence, dose jitter,
bandwidth and maturity: see notebook 05.